In [1]:
import pandas as pd
import os

In [2]:
try:
    df = pd.read_csv("../data/crime.csv")
    df.columns = df.columns.str.strip().str.upper()
    print("Dataset loaded!")
except FileNotFoundError:
    print("Error: crime.csv not found in ../data/")


Dataset loaded!


In [3]:
df.columns

Index(['STATE', 'DISTRICT', 'YEAR', 'MURDER', 'ATTEMPT TO MURDER',
       'CULPABLE HOMICIDE NOT AMOUNTING TO MURDER', 'RAPE', 'CUSTODIAL RAPE',
       'OTHER RAPE', 'KIDNAPPING & ABDUCTION',
       'KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS',
       'KIDNAPPING AND ABDUCTION OF OTHERS', 'DACOITY',
       'PREPARATION AND ASSEMBLY FOR DACOITY', 'ROBBERY', 'BURGLARY', 'THEFT',
       'AUTO THEFT', 'OTHER THEFT', 'RIOTS', 'CRIMINAL BREACH OF TRUST',
       'CHEATING', 'COUNTERFIETING', 'ARSON', 'HURT/GREVIOUS HURT',
       'DOWRY DEATHS', 'ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY',
       'INSULT TO MODESTY OF WOMEN', 'CRUELTY BY HUSBAND OR HIS RELATIVES',
       'IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES',
       'CAUSING DEATH BY NEGLIGENCE', 'OTHER IPC CRIMES', 'TOTAL IPC CRIMES'],
      dtype='str')

In [4]:
relevant_columns = [
    'STATE', 'DISTRICT', 'YEAR',
    # Women-specific (70%)
    'RAPE',
    'CUSTODIAL RAPE',
    'OTHER RAPE',
    'DOWRY DEATHS',
    'KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS',
    'CRUELTY BY HUSBAND OR HIS RELATIVES',
    'ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY',
    'INSULT TO MODESTY OF WOMEN',
    'IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES',
    # General violent (20%)
    'MURDER',
    'ATTEMPT TO MURDER',
    'CULPABLE HOMICIDE NOT AMOUNTING TO MURDER',
    'KIDNAPPING & ABDUCTION',
    'ROBBERY',
    'HURT/GREVIOUS HURT',
    # Property/Other (10%)
    'THEFT',
    'BURGLARY',
    'ARSON',
    'DACOITY',
    'TOTAL IPC CRIMES'
]


In [5]:
dist_df = df[relevant_columns].copy()

In [6]:
dist_df = dist_df[~dist_df['DISTRICT'].str.contains('TOTAL', case=False, na=False)]
if dist_df['YEAR'].nunique() > 1:
    print(f"Aggregating across {dist_df['YEAR'].nunique()} years...")
    dist_df = dist_df.groupby(['STATE', 'DISTRICT'], as_index=False).sum(numeric_only=True)


Aggregating across 12 years...


In [7]:
def calculate_risk(row):

    # WOMEN-SPECIFIC CRIMES → 70%
    women_score = (
        row['RAPE']                                                 * 0.15 +
        row['CUSTODIAL RAPE']                                       * 0.08 +
        row['OTHER RAPE']                                           * 0.07 +
        row['DOWRY DEATHS']                                         * 0.12 +
        row['KIDNAPPING AND ABDUCTION OF WOMEN AND GIRLS']          * 0.10 +
        row['CRUELTY BY HUSBAND OR HIS RELATIVES']                  * 0.08 +
        row['ASSAULT ON WOMEN WITH INTENT TO OUTRAGE HER MODESTY']  * 0.05 +
        row['INSULT TO MODESTY OF WOMEN']                           * 0.03 +
        row['IMPORTATION OF GIRLS FROM FOREIGN COUNTRIES']          * 0.02
    )  # = 70%

    # GENERAL VIOLENT CRIMES → 20%
    violent_score = (
        row['MURDER']                                       * 0.06 +
        row['ATTEMPT TO MURDER']                            * 0.04 +
        row['CULPABLE HOMICIDE NOT AMOUNTING TO MURDER']    * 0.03 +
        row['KIDNAPPING & ABDUCTION']                       * 0.04 +
        row['ROBBERY']                                      * 0.02 +
        row['HURT/GREVIOUS HURT']                           * 0.01
    )  # = 20%

    # PROPERTY / OTHER CRIMES → 10%
    property_score = (
        row['THEFT']    * 0.04 +
        row['BURGLARY'] * 0.03 +
        row['ARSON']    * 0.02 +
        row['DACOITY']  * 0.01
    )  # = 10%

    return women_score + violent_score + property_score

dist_df['RISK_SCORE'] = dist_df.apply(calculate_risk, axis=1)



In [8]:
high_threshold = dist_df['RISK_SCORE'].quantile(0.80)  # top 20% (HIGH)
low_threshold  = dist_df['RISK_SCORE'].quantile(0.40)  # bottom 40% →(LOW)


In [9]:
def get_risk_level(score):
    if score > high_threshold:
        return pd.Series(['HIGH', 'red'])
    elif score > low_threshold:
        return pd.Series(['MEDIUM', 'orange'])
    else:
        return pd.Series(['LOW', 'green'])

dist_df[['RISK_LEVEL', 'MARKER_COLOR']] = dist_df['RISK_SCORE'].apply(get_risk_level)


In [10]:
dist_df['STATE']    = dist_df['STATE'].str.title().str.strip()
dist_df['DISTRICT'] = dist_df['DISTRICT'].str.title().str.strip()


In [11]:
output_path = "../data/processed_crime_data.csv"
if not os.path.exists(output_path):
    dist_df.to_csv(output_path, index=False)
    print(f"Created: {output_path}")
else:
    print("File already exists, skipping save.")


Created: ../data/processed_crime_data.csv


In [12]:
print(f"\nTotal districts: {len(dist_df)}")
print(f"\nRisk distribution:\n{dist_df['RISK_LEVEL'].value_counts()}")
print(f"\nTop 5 HIGH risk districts:")
print(dist_df[dist_df['RISK_LEVEL'] == 'HIGH'][['STATE', 'DISTRICT', 'RISK_SCORE']]
      .sort_values('RISK_SCORE', ascending=False).head())


Total districts: 827

Risk distribution:
RISK_LEVEL
LOW       331
MEDIUM    330
HIGH      166
Name: count, dtype: int64

Top 5 HIGH risk districts:
              STATE           DISTRICT  RISK_SCORE
330       Karnataka   Bangalore Commr.     6544.36
463     Maharashtra      Mumbai Commr.     5371.87
15   Andhra Pradesh     Hyderabad City     4916.63
193         Gujarat   Ahmedabad Commr.     4332.44
797     West Bengal  24 Parganas North     3511.10
